# Fiche 3 — Parler à une API : requête, réponse, base64

| | |
|---|---|
| 🎯 **Objectif** | Lire et écrire une requête POST vers un modèle hébergé, puis décoder la réponse JSON jusqu'à un masque NumPy |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiches 1 et 2 |
| 🔗 **Dans le projet** | Étape 4 (script d'appel à l'API) |

> **L'idée clé.** Une API, c'est un **guichet** : on envoie une demande bien formée (adresse, identité, contenu), on reçoit une réponse avec un **code** (200 = tout va bien) et un contenu, ici du JSON.

## 1. Les quatre morceaux d'une requête

| Morceau | Ici | Analogie du courrier |
|---|---|---|
| **Adresse** (URL) | `https://router.huggingface.co/hf-inference/models/sayeed99/segformer_b3_clothes` | l'adresse du destinataire |
| **Méthode** | `POST` : on envoie quelque chose | « lettre » plutôt que « demande de renseignement » (`GET`) |
| **En-têtes** | `Authorization: Bearer hf_…` et `Content-Type: image/jpeg` | l'expéditeur et la nature du colis |
| **Corps** | les octets de l'image | le contenu du colis |

En Python, avec `requests`, tout tient en une ligne :

```python
response = requests.post(URL, headers={"Authorization": f"Bearer {token}", "Content-Type": "image/jpeg"},
                         data=octets_de_l_image, timeout=30)
```

⚠️ Le notebook fourni avec l'énoncé utilise l'adresse `api-inference.huggingface.co/models/…` : elle n'est plus en
service depuis 2025. L'adresse actuelle passe par `router.huggingface.co/hf-inference/…`.

## 2. Les codes de réponse à connaître

In [1]:
import pandas as pd

pd.DataFrame(
    [
        (200, "OK", "la réponse contient le résultat"),
        (400, "Bad Request", "la requête est mal formée (image illisible, mauvais Content-Type)"),
        (401, "Unauthorized", "token absent ou faux"),
        (402, "Payment Required", "plus de crédits sur le compte"),
        (404, "Not Found", "mauvaise adresse ou modèle inexistant"),
        (429, "Too Many Requests", "trop de requêtes : ralentir"),
        (503, "Service Unavailable", "modèle en cours de chargement ou serveur surchargé : réessayer plus tard"),
    ],
    columns=["code", "nom", "ce que ça veut dire ici"],
)

,code,nom,ce que ça veut dire ici
0,200,OK,la réponse contient le résultat
1,400,Bad Request,"la requête est mal formée (image illisible, ma..."
2,401,Unauthorized,token absent ou faux
3,402,Payment Required,plus de crédits sur le compte
4,404,Not Found,mauvaise adresse ou modèle inexistant
5,429,Too Many Requests,trop de requêtes : ralentir
6,503,Service Unavailable,modèle en cours de chargement ou serveur surch...


## 3. Une vraie réponse de l'API, décortiquée

Le projet a conservé une réponse réelle (`data/processed/exemple_reponse_api.json`) : on peut l'étudier **sans
token**. C'est une liste ; chaque élément décrit une classe détectée.

In [2]:
import json

from segvet import cache

reponse = json.loads((cache.PROCESSED_DIR / "exemple_reponse_api.json").read_text())["response"]
print(type(reponse).__name__, "de", len(reponse), "éléments")
premier = reponse[2]
print({cle: (valeur[:40] + "…" if isinstance(valeur, str) and len(valeur) > 40 else valeur) for cle, valeur in premier.items()})

list de 11 éléments
{'score': 1, 'label': 'Upper-clothes', 'mask': 'iVBORw0KGgoAAAANSUhEUgAAAVUAAAIACAAAAAAX…'}


Le champ `mask` est une **image PNG écrite en texte** (encodage *base64*) : JSON ne sait transporter que du texte,
alors on traduit les octets de l'image en lettres. Pour retrouver le masque : texte → octets → image → tableau.

In [3]:
import base64
import io

import numpy as np
from PIL import Image

octets = base64.b64decode(premier["mask"])
print("Les premiers octets :", octets[:8], "← signature d'un fichier PNG")
masque = np.array(Image.open(io.BytesIO(octets)))
print("Tableau :", masque.shape, masque.dtype, "valeurs", np.unique(masque).tolist())
print(f"{premier['label']} couvre {(masque > 0).mean():.1%} de l'image")

Les premiers octets : b'\x89PNG\r\n\x1a\n' ← signature d'un fichier PNG
Tableau : (512, 341) uint8 valeurs [0, 255]
Upper-clothes couvre 13.3% de l'image


Le masque d'une classe est en noir et blanc : 255 là où la classe est présente, 0 ailleurs. Il en faut un par classe ; la fiche 5 montre comment les fusionner en une seule carte.

## 4. Alléger l'image avant de l'envoyer

Le modèle redimensionne toute image en 512 × 512 avant de la traiter : envoyer plus grand ne fait qu'alourdir la
requête. On réduit donc l'image (plus grand côté ≤ 512 px) et on l'encode en JPEG, en mémoire, sans écrire de
fichier. Le `Content-Type` doit alors annoncer `image/jpeg`.

In [4]:
from segvet import io as segvet_io

chemin_png = segvet_io.load_pairs()[0][0]
image = Image.open(chemin_png).convert("RGB")
image.thumbnail((512, 512))                     # garde les proportions, plus grand côté ≤ 512
tampon = io.BytesIO()
image.save(tampon, format="JPEG", quality=90)   # encodage en mémoire
print(f"PNG d'origine : {chemin_png.stat().st_size / 1000:.0f} Ko | JPEG 512 px : {len(tampon.getvalue()) / 1000:.0f} Ko")

PNG d'origine : 294 Ko | JPEG 512 px : 34 Ko


Sur les 50 images du projet, le corrigé mesure 426 Ko contre 53 Ko en médiane, pour une qualité de segmentation quasi identique (0,005 point d'IoU de moins).

## ⚠️ Piège : lire l'image comme du texte

Une image se lit en mode **binaire** (`"rb"`). En mode texte (`"r"`), Python essaie de traduire les octets en
lettres… et échoue, ou pire, les abîme.

In [5]:
chemin = segvet_io.load_pairs()[0][0]
try:
    with open(chemin, "r", encoding="utf-8") as f:
        f.read()
except UnicodeDecodeError as erreur:
    print("Mode texte :", type(erreur).__name__, "→", str(erreur)[:70], "…")

with open(chemin, "rb") as f:
    donnees = f.read()
print(f"Mode binaire : {len(donnees)} octets, commence par {donnees[:4]}")

Mode texte : UnicodeDecodeError → 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte …
Mode binaire : 293914 octets, commence par b'\x89PNG'


### ✍️ À toi de jouer

À partir de `reponse`, construis un petit tableau pandas avec une ligne par classe détectée : son label et la part de l'image qu'elle couvre (en %), trié de la plus grande à la plus petite.

In [6]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
lignes = []
for element in reponse:
    m = np.array(Image.open(io.BytesIO(base64.b64decode(element["mask"]))))
    lignes.append({"classe": element["label"], "part (%)": 100 * (m > 0).mean()})
pd.DataFrame(lignes).sort_values("part (%)", ascending=False)
```

**Pourquoi ?** chaque masque se décode de la même façon ; on le transforme en booléen (`> 0`) puis la moyenne d'un tableau de booléens donne directement une proportion. Le fond arrive largement en tête, comme dans toutes les images du projet.

</details>

## ✅ À retenir

1. Une requête = adresse + méthode + en-têtes (identité, type de contenu) + corps (les octets de l'image).
2. Le **code de réponse** dit quoi faire : 200 on lit, 401/402/404 on corrige, 429/503 on attend et on réessaie.
3. La réponse est du **JSON** ; chaque masque y est une image PNG encodée en **base64**.
4. Les images se lisent en mode **binaire** (`"rb"`).
5. Vérifier l'adresse de l'API dans la documentation actuelle : elles changent (ici en 2025).